In [2]:
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    PyPDFLoader , 
    UnstructuredPDFLoader
)

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_22232\2471456568.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
d:\Code\AI\rag-mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# 1 loaders for PDF files by pypdfloader 
loader1 = PyPDFLoader("data/pdf/sample_paper.pdf")
documents1 = loader1.load()
print(f"Loaded {len(documents1)} documents using PyPDFLoader.")
print("-"*40)

print(documents1[0].page_content[:500])  # Print the first 500 characters of the first document
print("-"*40)
print("metadata:", documents1[0].metadata)
print("\n" + "="*80 + "\n")
loader2 = PyMuPDFLoader("data/pdf/sample_paper.pdf")
documents2 = loader2.load()
print(f"Loaded {len(documents2)} documents using PyMuPDFLoader.")
print("-"*40)
print(documents2[0].page_content[:500])  # Print the first 500 characters of the first document
print("-"*40)
print("metadata:", documents2[0].metadata)
print("\n" + "="*80 + "\n")



Loaded 11 documents using PyPDFLoader.
----------------------------------------
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
----------------------------------------
metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in a

In [ ]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class SmartPDFProcessor:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def _clean_text(self, text: str) -> str:
        # استبدال الحروف المشفرة وتنظيف المسافات
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        return " ".join(text.split()).strip()

    def process_pdf(self, pdf_path: str) -> List[Document]:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"الملف غير موجود: {pdf_path}")

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        total_pages = len(pages)
        processed_chunks: List[Document] = []

        for page_num, page in enumerate(pages, 1):
            clean_content = self._clean_text(page.page_content)

            # تخطي الصفحات شبه الفارغة (أقل من 50 حرف)
            if len(clean_content) < 50:
                print(f"⚠️ تم تخطي الصفحة {page_num} لقلة المحتوى ({len(clean_content)} حرف).")
                continue

            # إثراء الميتا-داتا
            enhanced_metadata = {
                **page.metadata,
                "page_number": page_num,
                "total_pages": total_pages,
                "chunk_method": "smart_recursive",
                "char_count": len(clean_content)
            }

            # إنشاء قطع Document مجزأة مع الميتا-داتا الجديدة
            chunks = self.text_splitter.create_documents(
                texts=[clean_content],
                metadatas=[enhanced_metadata]
            )
            processed_chunks.extend(chunks)

        return processed_chunks

print("✅ تم تعريف كلاس SmartPDFProcessor بنجاح!")